<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Assignment 4 — The Diet Problem at Your Restaurant

In Studio 3 you fed a football team at McDonald's. Now you do it for **your own restaurant**, the one
you were assigned in class. Same model, same notebook, a different menu.

**How this assignment works.** All of the code is already here. You do not write a model from
scratch. You:

1. point the notebook at your restaurant's menu file,
2. run it,
3. plug in scenarios in the cell marked **THE SCENARIO**, and
4. answer the questions in the cells marked **YOUR ANSWER**.

You are graded on your answers. They have to match **your** restaurant's numbers, so two students with
different restaurants will have different answers.

| Part | What you do | Points |
|---|---|---|
| 1 | The cheapest order at your restaurant | 20 |
| 2 | Three scenarios and a scoreboard | 25 |
| 3 | The shadow price of salt | 40 |
| 4 | A short memo to the athletic director | 15 |

**Write in full sentences, with units** (dollars per player, mg of sodium, and so on). A number with no
units and no sentence earns half credit.

**To turn it in:** make sure every cell has been run and its output is showing, then
File → Download → Download .ipynb and upload the file to HuskyCT.

## Import modules (download Pyomo)

Run this first, every time. It installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

# Part 0 · Your restaurant

Find your restaurant in this table and copy its file name into the cell below. The data is real:
nutrition from MenuStat (the New York City Health Department's restaurant nutrition database, 2018
edition) and menu prices as listed on fastfoodmenuprices.com in July and August 2026.

| Restaurant | File name |
|---|---|
| Arby's | `arbys_menu.csv` |
| Bojangles | `bojangles_menu.csv` |
| Burger King | `burger_king_menu.csv` |
| Chick-fil-A | `chick_fil_a_menu.csv` |
| Culver's | `culvers_menu.csv` |
| Dairy Queen | `dairy_queen_menu.csv` |
| Del Taco | `del_taco_menu.csv` |
| Hardee's | `hardees_menu.csv` |
| Jack in the Box | `jack_in_the_box_menu.csv` |
| Krystal | `krystal_menu.csv` |
| Panera Bread | `panera_bread_menu.csv` |
| Sonic | `sonic_menu.csv` |
| Starbucks | `starbucks_menu.csv` |
| Steak 'n Shake | `steak_n_shake_menu.csv` |
| Subway | `subway_menu.csv` |
| Taco Bell | `taco_bell_menu.csv` |
| Wendy's | `wendys_menu.csv` |
| Whataburger | `whataburger_menu.csv` |
| White Castle | `white_castle_menu.csv` |
| Zaxby's | `zaxbys_menu.csv` |

In [ ]:
# ---------- YOUR RESTAURANT (change these two lines) ----------
your_name = 'YOUR NAME HERE'
menu_file = 'mcdonalds_menu.csv'        # <-- change this to YOUR restaurant's file name from the table

# leave these alone
rules_file = 'assignment_rules.csv'
data_folder = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/menus/'

menu_df = pd.read_csv(data_folder + menu_file)
rules_df = pd.read_csv(data_folder + rules_file)

print("Name:", your_name)
print("Menu file:", menu_file, "-", len(menu_df), "items")
if menu_file == 'mcdonalds_menu.csv':
    print()
    print("STOP! This is still the McDonald's menu from class. Change menu_file to YOUR restaurant.")
menu_df

**The rules for this assignment** are per player, per day, like in class. They are a little looser
than the class rules (for example, sodium can go up to 9,000 mg) so that every restaurant has at least
one order that works.

In [ ]:
# the rules, per player per day
rules_df

# Part 1 · The cheapest order at your restaurant (20 points)

Run the next five cells without changing anything: **THE DATA**, the scoreboard, **THE SCENARIO**,
**THE MODEL**, and **SOLVE AND REPORT**. This is the same team order model from Studio 3.3:
53 players, one day, nobody gets more than 4 of the same item.

In [ ]:
# ---------- 1. THE DATA ----------
# the menu items - every column after 'price' is a nutrient
items = list(menu_df['item'])
nutrient_columns = list(menu_df.columns[3:])

# price[item] and nutrition[item][nutrient] (for ONE of that item)
price = {}
nutrition = {}
for i in range(len(menu_df)):
    item = menu_df.loc[i, 'item']
    price[item] = float(menu_df.loc[i, 'price'])
    nutrition[item] = {}
    for nutrient in nutrient_columns:
        nutrition[item][nutrient] = float(menu_df.loc[i, nutrient])

# a readable label for each nutrient (the rules file has nicer ones)
label = {}
for nutrient in nutrient_columns:
    label[nutrient] = nutrient

# the rules per player: one dictionary of minimums, one of maximums
player_min = {}
player_max = {}
for i in range(len(rules_df)):
    nutrient = rules_df.loc[i, 'nutrient']
    label[nutrient] = rules_df.loc[i, 'label']
    if rules_df.loc[i, 'rule'] == 'min':
        player_min[nutrient] = int(rules_df.loc[i, 'per_player'])
    else:
        player_max[nutrient] = int(rules_df.loc[i, 'per_player'])

print("minimums per player:", player_min)
print("maximums per player:", player_max)
print()
first_item = items[0]
print(first_item, ": $", price[first_item], nutrition[first_item])

In [ ]:
# an empty scoreboard: scenario name -> cost per player (run this cell ONCE)
scoreboard = {}

In [ ]:
# ---------- 2. THE SCENARIO (change things here) ----------
scenario = "1 base"
team_size = 53              # players on the roster
max_of_one_item = 4         # nobody gets more than 4 of the same item
low = dict(player_min)      # a fresh copy of the minimums per player
high = dict(player_max)     # a fresh copy of the maximums per player
banned = []                 # items nobody will eat
must_have = {}              # items everybody gets, and how many each

# Scenario 2: no variety rule (remove the # from the next two lines)
# scenario = "2 no variety rule"
# max_of_one_item = 100

# Scenario 3: coach's order - pick TWO items from YOUR menu that everybody gets one of.
# Copy the item names exactly as they appear in the menu table (spelling and capitals matter!)
# scenario = "3 coach's order"
# must_have = {'FIRST ITEM NAME' : 1, 'SECOND ITEM NAME' : 1}

# Scenario 4: your own idea. Some things you can change:
#   banned = ['AN ITEM NAME']           nobody eats this
#   low['protein'] = 200                more protein
#   high['sugar'] = 100                 less sugar
#   team_size = 11                      a different team
# scenario = "4 my own rule"

In [ ]:
# ---------- 3. THE MODEL (you only change the last part, if you want to) ----------
model = ConcreteModel()

# how many of each item to order for the whole team
model.x = Var(items, domain=NonNegativeReals)

# objective: the total bill
obj_expr = 0
for item in items:
    obj_expr += price[item] * model.x[item]

model.cost = Objective(
                      expr = obj_expr,
                      sense = minimize)

model.constraints = ConstraintList()

# the minimums (multiply by the team size!)
for nutrient in low:
    nutrient_expr = 0
    for item in items:
        nutrient_expr += nutrition[item][nutrient] * model.x[item]
    model.constraints.add(nutrient_expr >= low[nutrient] * team_size)

# the maximums
for nutrient in high:
    nutrient_expr = 0
    for item in items:
        nutrient_expr += nutrition[item][nutrient] * model.x[item]
    model.constraints.add(nutrient_expr <= high[nutrient] * team_size)

# variety: no more than max_of_one_item of any item per player
for item in items:
    model.constraints.add(model.x[item] <= max_of_one_item * team_size)

# scenario rules: banned items, and items everybody gets
for item in banned:
    model.constraints.add(model.x[item] == 0)
for item in must_have:
    model.constraints.add(model.x[item] >= must_have[item] * team_size)

# ---------- YOUR OWN CONSTRAINT (optional) ----------
# You can also write a constraint yourself, using item names from YOUR menu. For example:
# model.constraints.add(model.x['ONE ITEM NAME'] >= model.x['ANOTHER ITEM NAME'])


print("Number of variables =", len(model.x))
print("Number of constraints =", len(model.constraints))

In [ ]:
# ---------- 4. SOLVE AND REPORT (you never change this cell) ----------
results = SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

print("SCENARIO:", scenario)
if results.solver.termination_condition == TerminationCondition.optimal:
    per_player = round(model.cost() / team_size, 2)
    print("Total bill = $", round(model.cost(), 2))
    print("Per player = $", per_player)

    print()
    print("The order:")
    names = []
    counts = []
    for item in items:
        if model.x[item]() > 0.000001:
            print(item, ":", round(model.x[item](), 1), "total (", round(model.x[item]() / team_size, 2), "per player )")
            names.append(item)
            counts.append(model.x[item]() / team_size)

    print()
    print("Per player per day:")
    for nutrient in nutrient_columns:
        total = 0
        for item in items:
            total += nutrition[item][nutrient] * model.x[item]()
        rule_text = ""
        if nutrient in low:
            rule_text += " min " + str(low[nutrient])
        if nutrient in high:
            rule_text += " max " + str(high[nutrient])
        if rule_text == "":
            rule_text = " no rule"
        print(label[nutrient], ":", round(total / team_size, 1), "(" + rule_text + " )")

    # add this scenario to the scoreboard dictionary: scoreboard[key] = value
    scoreboard[scenario] = per_player

    # a picture of the order
    barh(names, counts, color='darkorange')
    xlabel('how many per player')
    title(scenario + ':  $%.2f per player' % per_player)
    tight_layout()
    savefig('fig_team_order.png')
    show()
else:
    print("INFEASIBLE! No order can satisfy every rule at the same time.")
    print("Loosen a rule and try again.")

### YOUR ANSWER - Part 1

Double-click this cell and type your answers.

**1.1 (5 points)** What is the cheapest bill per player, and for the whole team?

*your answer*

**1.2 (5 points)** Describe the order. Which items, and about how many of each per player?

*your answer*

**1.3 (5 points)** Which rules are **binding**? (Look at "Per player per day". A rule is binding when
the number sits exactly on its min or its max.)

*your answer*

**1.4 (5 points)** Would you want to eat this for a day? Name one thing a real team would care about
that this model leaves out.

*your answer*

# Part 2 · Three scenarios and a scoreboard (25 points)

Go back up to **THE SCENARIO** cell. For each scenario: remove the `#` from that scenario's lines, fill
in the item names if it asks for them, then re-run the scenario cell, the model cell, and the solve
cell. Put the `#` back before you try the next one. The scoreboard remembers every scenario you run.

| Scenario | What changes |
|---|---|
| 1 base | nothing (you already ran it) |
| 2 no variety rule | `max_of_one_item = 100` |
| 3 coach's order | everybody gets one each of two items **you choose** from your menu |
| 4 my own rule | your idea: ban an item, change a limit, change the team |

If a scenario comes back **INFEASIBLE**, that is a real answer, not a mistake. Say so, and say which
rule you would loosen.

When all four are on the scoreboard, run the cell below to draw it.

In [ ]:
# draw the scoreboard
scenario_names = []
dollars = []
for name in scoreboard:
    scenario_names.append(name)
    dollars.append(scoreboard[name])

barh(scenario_names, dollars, color='steelblue')
xlabel('$ per player')
title('Feed the team: what did each rule cost?')
gca().invert_yaxis()
tight_layout()
savefig('fig_scoreboard.png')
show()

### YOUR ANSWER - Part 2

**2.1 (8 points)** Fill in the table.

| Scenario | Cost per player | What changed in the order? |
|---|---|---|
| 1 base | | |
| 2 no variety rule | | |
| 3 coach's order | | |
| 4 my own rule | | |

**2.2 (5 points)** How much does the variety rule cost per player at your restaurant? What does the
order look like without it?

*your answer*

**2.3 (6 points)** Which two items did the coach order, and what did that cost per player compared
with the base order?

*your answer*

**2.4 (6 points)** What was your own rule, why did you pick it, and what did it cost the team?

*your answer*

# Part 3 · The shadow price of salt (40 points)

This is the most important part of the assignment. Read it slowly.The sodium rule is a **maximum**: each player may have at most so many milligrams of sodium a day.
When that rule is binding, it is costing the team money, because the cheapest foods are salty.

> **The shadow price of salt** is the answer to one question:
> **if the sodium limit were 100 mg higher, how many dollars per player would the order get cheaper?**

Read it the other way and it is just as useful: **what does it cost, per player, to cut 100 mg of
sodium?**

We find it exactly the way we found the value of a carpentry hour. Solve the model at one sodium
limit, solve it again with the limit 100 mg higher, and subtract.

Three things to be clear about:

1. **The units are dollars per player, per 100 mg of sodium.** For the whole team, multiply by the team size.
2. **It is not one number.** It depends on where the limit is. A tight limit has a high shadow price;
   a loose limit has a shadow price of zero.
3. **Zero means the rule is not binding.** Above some limit the cheapest order does not even use all
   the sodium it is allowed, so allowing more saves nothing. Diamonds to sand.

Now the loop. It tries every sodium limit from 2,000 mg to 10,000 mg, 100 mg at a time (81 solves, so
give it a little while). It is your team order model with **one change**: the sodium limit is `a`.
Every other rule stays at the assignment value.

In [ ]:
# ---------- THE SCENARIO (change things here) ----------
myRange = np.arange(2000, 10100, 100)    # the sodium limits to try, in steps of 100 mg
team_size = 53
max_of_one_item = 4

In [ ]:
# ---------- THE LOOP (you never change this cell) ----------
saltResults = pd.DataFrame()

for a in myRange:
    low = dict(player_min)
    high = dict(player_max)

    # the ONE thing we change each time through the loop: the sodium limit
    high['sodium'] = int(a)

    # the model (same as the team order notebook)
    model = ConcreteModel()
    model.x = Var(items, domain=NonNegativeReals)
    obj_expr = 0
    for item in items:
        obj_expr += price[item] * model.x[item]
    model.cost = Objective(expr = obj_expr, sense = minimize)
    model.constraints = ConstraintList()
    for nutrient in low:
        nutrient_expr = 0
        for item in items:
            nutrient_expr += nutrition[item][nutrient] * model.x[item]
        model.constraints.add(nutrient_expr >= low[nutrient] * team_size)
    for nutrient in high:
        nutrient_expr = 0
        for item in items:
            nutrient_expr += nutrition[item][nutrient] * model.x[item]
        model.constraints.add(nutrient_expr <= high[nutrient] * team_size)
    for item in items:
        model.constraints.add(model.x[item] <= max_of_one_item * team_size)

    results = SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

    # store the cost per player (if there is no feasible order, store a blank)
    if results.solver.termination_condition == TerminationCondition.optimal:
        cost_per_player = model.cost() / team_size
    else:
        cost_per_player = np.nan
    myX = pd.DataFrame([a, cost_per_player])
    myX = myX.T
    saltResults = pd.concat([saltResults, myX])

saltResults = saltResults.rename({0:"Sodium limit (mg)", 1:"Cost per player ($)"}, axis='columns')
saltResults.reset_index(drop=True, inplace=True)
saltResults

## The shadow price, one subtraction at a time

Each row of the table is 100 mg more sodium than the row above it. So:

**shadow price of salt = (cost in the row above) − (cost in this row)**

That is how many dollars per player we save when the limit goes up by 100 mg. The next cell does the
subtraction for every row. (`diff()` gives this row minus the row above, which is negative because the
cost is falling, so we flip the sign.)

In [ ]:
# dollars saved per player when the sodium limit goes up by 100 mg
saltResults['Shadow price of salt ($ per 100 mg)'] = -saltResults['Cost per player ($)'].diff()

# show the whole table
pd.set_option('display.max_rows', 100)
saltResults.round(3)

In [ ]:
# picture 1: the bill
saltResults.plot(x='Sodium limit (mg)', y='Cost per player ($)')
title('Cost per player at each sodium limit')
show()

# picture 2: the shadow price
saltResults.plot(x='Sodium limit (mg)', y='Shadow price of salt ($ per 100 mg)', color='darkorange')
title('The shadow price of salt')
show()

## How to read it

Look at the second picture from right to left, starting with a loose limit and tightening it:

- **Flat at \$0 on the right.** The limit is loose. The cheapest order does not use all the sodium it
  is allowed, so the rule is **not binding** and 100 mg more is worth nothing.
- **Then it steps up.** Now the rule is binding. Every 100 mg you take away forces the order toward
  less salty, more expensive food. The height of the step is what each 100 mg costs per player.
- **It climbs as the limit gets tighter.** The cheap low-sodium options run out, so each cut hurts
  more than the last one.
- **Then the line stops.** Blanks (`NaN`) in the table mean **infeasible**: no order from this menu can
  meet every rule at that sodium limit, at any price.

**Saying it out loud.** If the shadow price at a 5,500 mg limit is \$0.33, then: *"At a 5,500 mg
limit, letting each player have 100 mg more sodium would save 33 cents per player, or about \$17.50 for
a team of 53. Cutting 100 mg would cost that much."*

### YOUR ANSWER - Part 3

Use the table and the two pictures. Every answer needs a number from **your** table, with units.

**3.1 (6 points)** What is the **lowest sodium limit** your restaurant can meet (to the nearest
100 mg)? What does the table show below that limit, and what does it mean?

*your answer*

**3.2 (8 points)** The limit in this assignment is **9,000 mg**. What is the shadow price of salt at
9,000 mg for your restaurant? Is the sodium rule binding there? How can you tell?

*your answer*

**3.3 (8 points)** In class we used **7,000 mg**. What is the shadow price of salt at 7,000 mg for
your restaurant? Write it as a full sentence with units, like the example above. (If your table shows
a blank at 7,000 mg, say what that means for your restaurant.)

*your answer*

**3.4 (6 points)** Tighten the limit from loose to tight. At what limit does the shadow price **first
rise above \$0**? Why is it \$0 above that point? (If it never gets down to \$0 in your table, say so,
and say what that tells you about your restaurant.)

*your answer*

**3.5 (6 points)** The team doctor wants to cut the limit from 9,000 mg to **1,000 mg above your
lowest feasible limit** (your answer to 3.1 plus 1,000). What does that cost per player? For the whole
team of 53?

*your answer*

**3.6 (6 points)** In two sentences, explain "the shadow price of salt" to someone who has never taken
this class. No jargon.

*your answer*

# Part 4 · Memo to the athletic director (15 points)

Write a short memo (about 150 words) in the cell below. It should say:

- what you recommend ordering, and what it costs per player and for the team,
- which rule costs the team the most money, with a number to back it up, and
- **one recommendation about the sodium limit**, backed by a shadow price from Part 3.

The athletic director is not a coder and has never heard of Pyomo. Write for that reader.

### YOUR ANSWER - Part 4

*your memo*

## Before you turn it in

- Your name and **your** restaurant's file are in the Part 0 cell (not McDonald's).
- Every cell has been run and its output is showing.
- Your scoreboard chart shows four scenarios. (Careful: **Runtime → Run all** starts the scoreboard
  over, so do not use it at the very end.)
- Every **YOUR ANSWER** cell is filled in, in full sentences, with units.
- File → Download → Download .ipynb, then upload that file to HuskyCT.